# Week 9 demos

In [ ]:
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

## Example 1: CalCOFI CTD / bottle cast

In this example, consider the quality-controlled oceanographic CTD and bottle dataset from the California Cooperative Oceanic Fisheries Investigations (CalCOFI). Our particular subset is from the July 2020 CalCOFI cruise conducted aboard the research vessel Sally Ride, and the data were collected during the downcast.

In [ ]:
# Use urlretrieve to obtain the csv file
file_url = 'https://raw.githubusercontent.com/OCEAN-215-2026/week_09_demos/refs/heads/main/data/20-2007SR_CTDBTL_001-075D.csv'
#urlretrieve(file_url, "data/20-2007SR_CTDBTL_001-075D.csv")

In [ ]:
# Load the CalCOFI CTD downcast data from July 2020
CalCOFI = pd.read_csv("data/20-2007SR_CTDBTL_001-075D.csv")
CalCOFI.info()

<!-- BEGIN QUESTION -->

### Poll #1 (JupyterHub **on**)

This cruise collected samples from a number of distinct stations, identified by their station ID `sta_ID`. How many stations were there?

+ 13
+ 75
+ 739
+ 30071

In [ ]:
# Your code here


<!-- END QUESTION -->

Now let's focus on the deepest station. To identify the deepest station, we apply `.groupby()` with `.max()` on the `Depth` column

In [ ]:
# Use groupby to find the maximum depth of each station

CalCOFI_bySta = ...

display(CalCOFI_bySta)

In [ ]:
# Extract the station whose maximum depth is the maximum (i.e., the deepest station)
CalCOFI_bySta.loc[CalCOFI_bySta["Depth"] == CalCOFI_bySta["Depth"].max()]

In [ ]:
# subset to the station with the deepest cast, and pick 3 variables to look at
Sta_87_40 = CalCOFI.loc[
    CalCOFI["Sta_ID"] == "086.7 040.0", 
    ["Depth", "TempAve", "SaltAve_Corr", "pH"]
]

In [ ]:
# Calculate the binned depth

Sta_87_40["Binned_Depth"] = 10 * (Sta_87_40["Depth"].astype(int) // 10) + 5

display(Sta_87_40)

<!-- BEGIN QUESTION -->

### Your turn #1

Calculate the bin-averaged temperature, salinity, and pH using a single group-by summarization. Call the resulting pandas dataframe `Sta_87_40_binned`

In [ ]:
# Your code here


<!-- END QUESTION -->



In [ ]:
# let's plot one variable to see how it looks

fig = plt.figure()
ax = fig.add_subplot()

ax.plot(Sta_87_40_binned["SaltAve_Corr"], Sta_87_40_binned["Binned_Depth"])

ax.set_xlabel("Salinity (PSU)")
ax.set_ylabel("Depth (m)")
ax.set_title("10-m bin-averaged Salinity at Station 086.7 040.0")

ax.invert_yaxis()

plt.show(fig)

In [ ]:
# Compare with the plot using raw data

fig = plt.figure()
ax = fig.add_subplot()

ax.plot(Sta_87_40["SaltAve_Corr"], Sta_87_40["Depth"])

ax.set_xlabel("Salinity (PSU)")
ax.set_ylabel("Depth (m)")
ax.set_title("1-m interval Salinity at Station 086.7 040.0")

ax.invert_yaxis()

plt.show(fig)

## Example 2: climatology and anomaly

In this second example we'll consider same sea ice concentration data that we considered in week 8. As with week 8 we will focus on the time dependence and thus we'll reduce the spatial dimensions.

In [ ]:
# Use urlretrieve to obtain the csv file
file_url = 'https://raw.githubusercontent.com/OCEAN-215-2026/week_09_demos/refs/heads/main/data/icec.mnmean.nc'
#urlretrieve(file_url, "data/20-2007SR_CTDBTL_001-075D.csv")

In [ ]:
# same dataset, same subset and dimension reduction as in week 8

data = xr.open_dataset('data/icec.mnmean.nc')
ice = data["icec"]

ice_mean = ice.sel(lat = ice["lat"] > 45).mean(['lat', 'lon'])

ice_df = ice_mean.to_dataframe().reset_index()

display(ice_df)

<!-- BEGIN QUESTION -->

### Your turn #2

Create a new column called `month` that store the month of the ice thickness measurement

In [ ]:
# Your code here


<!-- END QUESTION -->



In [ ]:
# Now we can compute the ice thickness climatology
# by taking the mean over all years for a fixed month (Jan, Feb, etc.)

ice_mean_climatology = ice_df.loc[:, ["month", "icec"]].groupby("month").mean()
ice_mean_climatology

In [ ]:
# Plot of the climatology

fig = plt.figure(figsize = (10, 4))
ax = fig.add_subplot()

ax.plot(ice_mean_climatology, lw=2)
ax.set_xlabel('Month')
ax.set_ylabel('Sea Ice Concentration (%)')
ax.set_title('Arctic Averaged Sea Ice Concentration Mean Seasonal Cycle')
ax.grid()

plt.show(fig)

In [ ]:
# Next we calculate Anomalies
# These are deviations from the mean seasonal cycle

# for each row in ice_df, we want to extract the corresponding long-term mean of its ice concentration
# This is achieved using ice_mean_climatology.loc[ice_df["month"]]
# But pandas perform arithmetics by aligning indices, which would be wrong in our case
# To prevent that we extract the internal numpy array via .values

ice_df["anomalies"] = ice_df["icec"] - ice_mean_climatology.loc[ice_df["month"], "icec"].values
ice_df

In [ ]:
# Plot of the anomalies

fig = plt.figure(figsize = (10, 4))
ax = fig.add_subplot()

ax.plot(ice_df["time"], ice_df["anomalies"], lw=2)
ax.set_xlabel('Month')
ax.set_ylabel('Sea Ice Concentration (%)')
ax.set_title('Arctic Averaged Sea Ice Concentration Mean Seasonal Cycle')
ax.grid()

plt.show(fig)

<!-- BEGIN QUESTION -->

### Poll #2 (JupyterHub **on**)

Perform a linear regression on the anomalies. What portion of the variance is explained by the linear trend?

+ 33%
+ 44%
+ 56%
+ 67%

In [ ]:
# Your code here


<!-- END QUESTION -->



In [ ]:
# Finally, let's visualize the best-fitted line
